<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-20 23:32:43
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.54 C
CAGR/XIRR %:     1.45%
Est FTT Value:   2.74 C
Future Upside:   1.19 C (77.24%)
=================== PERFORMANCE ==========================
Current PnL:             -40.85 L (-21.28%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    8.38 L (5.62%)
CY Booked PnL:           6.36 L (4.12%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               40.51 K (0.26%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             3.14 L
Total Loss:               -43.99 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
50,M&M,1.03,10.68,6.15,17.48,20494.0,32155.0,333230.0,3762.88,-2.07,74.0,X-LC,4.13,25.0,1.57,2.16,28.85,X40N,ATH,AUTO
12,BOSCH-HCIL,2.14,-21.09,26.71,-0.01,26937.0,-26955.0,100850.0,2282.00,792.28,56.0,M-SC,18.60,231.0,-1.00,0.65,71.66,OX40N,BTT,AC
75,TTKPRESTIG,-2.77,-29.82,42.62,0.09,30144.0,-30050.0,70727.0,770.00,-28.95,47.0,M-SC,2.94,250.0,-1.00,0.46,27.30,OX40N,NTT,DURABLES
76,UNITDSPR,0.44,7.72,12.11,20.77,30544.0,18077.0,252221.0,1644.00,-5.70,63.0,X-MC,0.64,64.0,0.59,1.63,21.24,X40N,NTT,BREWERIES
52,MASFIN,-1.97,-21.51,54.48,21.26,41898.0,-21075.0,76905.0,396.02,-39.43,43.0,H-SC,14.48,167.0,-0.50,0.50,4.78,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,JIOFIN,-0.81,-20.89,72.02,36.09,228777.0,-83887.0,317658.0,387.00,-29.75,54.0,X-LC,7.66,34.0,-0.37,2.06,10.01,XY24,BTT,FINANCE
44,INFY,0.56,-20.56,79.60,42.66,253462.0,-82435.0,318420.0,1972.00,-42.12,46.0,X-LC,6.40,3.0,-0.33,2.06,13.99,X40,NTT,IT
38,HYUNDAI,-0.19,13.26,25.63,42.29,82317.0,37604.0,321175.0,2782.73,32.10,66.0,X-LC,2.13,29.0,0.46,2.08,32.21,X40N,ATH,AUTO
33,HCLTECH,0.86,-8.01,36.65,25.70,121703.0,-28906.0,332067.0,1837.07,-14.30,45.0,X-LC,9.23,8.0,-0.24,2.15,32.49,X40,ATH,IT
23,DMART,-0.29,-12.86,54.48,34.61,184431.0,-49979.0,338530.0,5391.45,-28.56,57.0,X-LC,14.74,14.0,-0.27,2.19,11.36,X40N,ATH,FMCG


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
43,INDUSINDBK,8.59,-25.95,79.46,32.89,45428.0,-20036.0,57171.0,1800.00,84.92,70.0,L-MC,3.60,258.0,-0.44,0.37,33.51,XR,NTT,BANKS
70,TATAELXSI,3.36,-44.51,137.95,32.03,100910.0,-58685.0,73150.0,9161.00,-36.54,59.0,H-SC,11.13,158.0,-0.58,0.47,28.50,OX40N,NTT,IT
1,ABBOTINDIA,3.23,-11.02,29.65,15.37,47188.0,-19705.0,159150.0,34390.26,-28.60,54.0,X-MC,0.57,58.0,-0.42,1.03,7.81,X40,ATH,PHARMA
8,BAJAJHLDNG,3.16,4.47,21.62,27.06,75685.0,14983.0,350070.0,14191.91,-9.39,64.0,X-LC,3.63,28.0,0.20,2.27,35.86,X40,ATH,FINANCE
51,MARUTI,3.11,3.11,22.19,25.99,77726.0,10565.0,350275.0,17120.06,6.27,76.0,X-LC,8.65,33.0,0.14,2.27,25.43,XR,ATH,AUTO


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
79,VALIANTORG,-4.98,-32.84,159.32,74.15,175046.0,-53734.0,109871.0,838.0,NaN,34.0,NaN,34.53,145.0,-0.31,0.71,63.50,XR,NTT,CHEMICALS
75,TTKPRESTIG,-2.77,-29.82,42.62,0.09,30144.0,-30050.0,70727.0,770.0,-28.95,47.0,M-SC,2.94,250.0,-1.00,0.46,27.30,OX40N,NTT,DURABLES
56,RAJESHEXPO,-2.42,-90.76,982.78,0.07,124597.0,-124499.0,12678.0,518.0,NaN,30.0,NaN,47.79,268.0,-1.00,0.08,0.00,OX40N,NTT,JEWELLERY
29,GREENPANEL,-2.40,-59.51,253.50,43.14,187925.0,-108946.0,74132.0,537.0,NaN,57.0,NaN,23.03,225.0,-0.58,0.48,22.02,XY24,NTT,MISC
49,LAOPALA,-2.32,-52.95,195.60,39.09,120664.0,-69416.0,61689.0,464.0,NaN,54.0,NaN,13.46,154.0,-0.58,0.40,12.90,AR,NTT,CERAMICS


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.89,51.99,35.83,106.44,80403.0,76756.0,224400.0,1629.94,-4.99,46.0,M-SC,22.03,220.0,0.95,1.45,70.03,OX40N,ATH,CABLES
65,SIS,-0.99,-17.38,46.77,21.26,42905.0,-19296.0,91736.0,528.00,76.32,34.0,H-SC,5.36,168.0,-0.45,0.59,36.40,OX40N,NTT,MISC
12,BOSCH-HCIL,2.14,-21.09,26.71,-0.01,26937.0,-26955.0,100850.0,2282.00,792.28,56.0,M-SC,18.60,231.0,-1.00,0.65,71.66,OX40N,BTT,AC
42,INDIGOPNTS,-1.08,-29.55,41.97,0.02,51613.0,-51582.0,122977.0,1408.00,-32.85,51.0,M-SC,6.69,222.0,-1.00,0.80,40.12,OX40N,NTT,PAINTS
75,TTKPRESTIG,-2.77,-29.82,42.62,0.09,30144.0,-30050.0,70727.0,770.00,-28.95,47.0,M-SC,2.94,250.0,-1.00,0.46,27.30,OX40N,NTT,DURABLES


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.89,51.99,35.83,106.44,80403.0,76756.0,224400.0,1629.94,-4.99,46.0,M-SC,22.03,220.0,0.95,1.45,70.03,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.89,51.99,35.83,106.44,80403.0,76756.0,224400.0,1629.94,-4.99,46.0,M-SC,22.03,220.0,0.95,1.45,70.03,OX40N,ATH,CABLES
66,STARHEALTH,-2.22,7.60,36.86,47.26,111498.0,21368.0,302491.0,761.00,-21.54,52.0,H-SC,3.37,173.0,0.19,1.96,30.79,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-0.89,51.99,35.83,106.44,80403.0,76756.0,224400.0,1629.94,-4.99,46.0,M-SC,22.03,220.0,0.95,1.45,70.03,OX40N,ATH,CABLES
83,WIPRO,0.96,-14.66,126.17,93.01,269442.0,-36685.0,213555.0,420.00,-26.53,47.0,M-LC,7.95,98.0,-0.14,1.38,18.44,XR,NTT,IT
69,TANLA,-1.40,-25.34,188.21,115.17,380455.0,-68617.0,202144.0,1903.93,-18.77,57.0,H-SC,25.10,147.0,-0.18,1.31,78.70,AR,ATH,IT
41,INDIAMART,-0.93,-33.90,203.99,100.92,166291.0,-41817.0,81519.0,4675.60,-60.70,44.0,H-SC,16.56,140.0,-0.25,0.53,3.42,AR,ATH,MISC
3,ALKYLAMINE,-0.86,-33.59,189.95,92.54,127354.0,-33917.0,67046.0,4520.85,-44.15,35.0,H-SC,7.99,142.0,-0.27,0.43,27.31,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,-0.14,-19.09,63.41,32.21,137688.0,-51239.0,217140.0,50690.44,-38.63,31.0,X-SC,6.60,55.0,-0.37,1.41,4.19,X40,ATH,APPARELS
73,TMPV,1.08,-33.59,125.27,49.60,396718.0,-160182.0,316690.0,600.00,-539.43,43.0,X-MC,17.21,4.0,-0.40,2.05,5.46,XY24,NTT,AUTO
37,HONAUT,1.13,-21.32,91.77,50.88,139238.0,-41113.0,151725.0,58192.80,-38.31,44.0,X-SC,9.10,90.0,-0.30,0.98,15.60,X40N,ATH,ELECTRICAL
26,GILLETTE,-0.88,-16.22,53.60,28.69,114015.0,-41181.0,212715.0,10891.01,-40.01,44.0,X-SC,6.17,83.0,-0.36,1.38,2.78,X40,ATH,FMCG
33,HCLTECH,0.86,-8.01,36.65,25.70,121703.0,-28906.0,332067.0,1837.07,-14.30,45.0,X-LC,9.23,8.0,-0.24,2.15,32.49,X40,ATH,IT


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,3.23,-11.02,29.65,15.37,47188.0,-19705.0,159150.0,34390.26,-28.60,54.0,X-MC,0.57,58.0,-0.42,1.03,7.81,X40,ATH,PHARMA
76,UNITDSPR,0.44,7.72,12.11,20.77,30544.0,18077.0,252221.0,1644.00,-5.70,63.0,X-MC,0.64,64.0,0.59,1.63,21.24,X40N,NTT,BREWERIES
38,HYUNDAI,-0.19,13.26,25.63,42.29,82317.0,37604.0,321175.0,2782.73,32.10,66.0,X-LC,2.13,29.0,0.46,2.08,32.21,X40N,ATH,AUTO
17,CIPLA,-0.25,6.36,16.48,23.89,52701.0,19135.0,319790.0,1648.14,39.87,61.0,X-MC,2.22,18.0,0.36,2.07,19.78,X40N,ATH,PHARMA
11,BERGEPAINT,-0.35,-14.41,46.41,25.32,102387.0,-37135.0,220614.0,680.00,-23.58,45.0,X-MC,2.61,68.0,-0.36,1.43,15.69,XY24,NTT,PAINTS


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
26,GILLETTE,-0.88,-16.22,53.60,28.69,114015.0,-41181.0,212715.0,10891.01,-40.01,44.0,X-SC,6.17,83.0,-0.36,1.38,2.78,X40,ATH,FMCG
54,PAGEIND,-0.14,-19.09,63.41,32.21,137688.0,-51239.0,217140.0,50690.44,-38.63,31.0,X-SC,6.60,55.0,-0.37,1.41,4.19,X40,ATH,APPARELS
6,AWL,0.46,-40.78,182.01,67.02,384076.0,-145290.0,211019.0,485.00,-73.96,51.0,X-SC,10.68,81.0,-0.38,1.37,4.96,XY24,NTT,FMCG
73,TMPV,1.08,-33.59,125.27,49.60,396718.0,-160182.0,316690.0,600.00,-539.43,43.0,X-MC,17.21,4.0,-0.40,2.05,5.46,XY24,NTT,AUTO
55,PGHH,1.42,-47.40,149.07,31.02,167435.0,-101200.0,112320.0,17484.41,-60.53,50.0,X-SC,22.23,57.0,-0.60,0.73,5.74,X40,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
10,BATAINDIA,1.16,-59.07,230.34,35.20,121306.0,-76006.0,52664.0,2096.00,-18.29,49.0,X-SC,6.49,91.0,-0.63,0.34,9.44,X40,NTT,FOOTWEAR
55,PGHH,1.42,-47.40,149.07,31.02,167435.0,-101200.0,112320.0,17484.41,-60.53,50.0,X-SC,22.23,57.0,-0.60,0.73,5.74,X40,ATH,FMCG
37,HONAUT,1.13,-21.32,91.77,50.88,139238.0,-41113.0,151725.0,58192.80,-38.31,44.0,X-SC,9.10,90.0,-0.30,0.98,15.60,X40N,ATH,ELECTRICAL
2,ACC,0.66,-42.85,205.87,74.81,315475.0,-114888.0,153240.0,3906.00,22.41,66.0,X-SC,11.21,85.0,-0.36,0.99,13.52,XY24,BTT,CEMENT
1,ABBOTINDIA,3.23,-11.02,29.65,15.37,47188.0,-19705.0,159150.0,34390.26,-28.60,54.0,X-MC,0.57,58.0,-0.42,1.03,7.81,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,-0.63,-14.83,51.12,28.70,190514.0,-64916.0,372679.0,452.00,-19.58,72.0,X-LC,5.52,1.0,-0.34,2.41,17.29,X40,NTT,FMCG
71,TCS,1.43,-28.43,77.10,26.75,269311.0,-138774.0,349301.0,4208.34,-42.67,47.0,X-LC,6.91,2.0,-0.52,2.26,21.15,X40,ATH,IT
44,INFY,0.56,-20.56,79.60,42.66,253462.0,-82435.0,318420.0,1972.00,-42.12,46.0,X-LC,6.40,3.0,-0.33,2.06,13.99,X40,NTT,IT
73,TMPV,1.08,-33.59,125.27,49.60,396718.0,-160182.0,316690.0,600.00,-539.43,43.0,X-MC,17.21,4.0,-0.40,2.05,5.46,XY24,NTT,AUTO
33,HCLTECH,0.86,-8.01,36.65,25.70,121703.0,-28906.0,332067.0,1837.07,-14.30,45.0,X-LC,9.23,8.0,-0.24,2.15,32.49,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
43,INDUSINDBK,8.59,-25.95,79.46,32.89,45428.0,-20036.0,57171.0,1800.00,84.92,70.0,L-MC,3.60,258.0,-0.44,0.37,33.51,XR,NTT,BANKS
12,BOSCH-HCIL,2.14,-21.09,26.71,-0.01,26937.0,-26955.0,100850.0,2282.00,792.28,56.0,M-SC,18.60,231.0,-1.00,0.65,71.66,OX40N,BTT,AC
79,VALIANTORG,-4.98,-32.84,159.32,74.15,175046.0,-53734.0,109871.0,838.00,NaN,34.0,NaN,34.53,145.0,-0.31,0.71,63.50,XR,NTT,CHEMICALS
69,TANLA,-1.40,-25.34,188.21,115.17,380455.0,-68617.0,202144.0,1903.93,-18.77,57.0,H-SC,25.10,147.0,-0.18,1.31,78.70,AR,ATH,IT
27,GLAXO,0.52,9.50,23.92,35.69,52195.0,18924.0,218208.0,3380.16,-20.48,59.0,X-MC,6.26,56.0,0.36,1.41,29.69,X40,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
51,MARUTI,3.11,3.11,22.19,25.99,77726.0,10565.0,350275.0,17120.06,6.27,76.0,X-LC,8.65,33.0,0.14,2.27,25.43,XR,ATH,AUTO
81,VOLTAS,-0.16,-3.39,50.28,45.20,93144.0,-6492.0,185250.0,1856.00,5.40,72.0,X-MC,15.14,79.0,-0.07,1.20,30.52,XY25,NTT,AC
50,M&M,1.03,10.68,6.15,17.48,20494.0,32155.0,333230.0,3762.88,-2.07,74.0,X-LC,4.13,25.0,1.57,2.16,28.85,X40N,ATH,AUTO
34,HDFCBANK,2.94,6.20,26.83,34.70,103050.0,22432.0,384084.0,1073.00,-18.62,78.0,X-LC,5.29,26.0,0.22,2.49,24.38,X40,BTT,BANKS
69,TANLA,-1.40,-25.34,188.21,115.17,380455.0,-68617.0,202144.0,1903.93,-18.77,57.0,H-SC,25.10,147.0,-0.18,1.31,78.70,AR,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,23.03
1,25,51.97
2,50,82.61


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    39.31
SC    28.18
MC    21.71
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.26
XY24     22.62
X40N     19.49
XY25     10.21
XR        9.87
OX40N     6.63
AR        3.17
SR        0.70
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    37.93
X-MC    20.28
H-SC    13.88
X-SC     8.75
M-SC     5.55
M-LC     1.38
M-MC     1.06
L-MC     0.37
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.62,-22.51,66.70
IT,12.32,-34.68,104.85
FINANCE,10.94,-20.75,72.14
AUTO,9.43,-9.79,50.13
PAINTS,5.58,-22.95,45.04
MISC,5.50,-45.14,103.48
INSURANCE,5.43,1.52,35.49
BANKS,5.37,-8.11,62.32
PHARMA,4.51,2.63,21.82


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4001482.0,19
X40,2122416.0,16
XR,1579553.0,13
X40N,1421475.0,12
AR,1019558.0,6
OX40N,763957.0,10
XY25,713421.0,6
SR,304439.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2393671.0,17
H-SC,2300753.0,15
X-MC,1831008.0,14
X-SC,1453604.0,8
M-SC,514569.0,7
M-LC,269442.0,1
M-MC,81338.0,1
L-MC,45428.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1252976.0      8
H-SC       XY24       995408.0      5
X-SC       XY24       699551.0      2
X-MC       XY24       664936.0      3
           X40N       557126.0      5
H-SC       AR         546746.0      2
X-SC       X40        540444.0      4
X-LC       X40N       482059.0      4
H-SC       XR         382197.0      4
X-LC       XY25       352133.0      3
X-MC       X40        328996.0      4
           XY25       279950.0      2
M-LC       XR         269442.0      1
H-SC       OX40N      249048.0      3
X-LC       XY24       228777.0      1
X-SC       X40N       213609.0      2
M-SC       XR         194236.0      2
           OX40N      189097.0      4
           XY24       131236.0      1
H-SC       SR         127354.0      1
M-MC       XY25        81338.0      1
X-LC       XR          77726.0      1
L-MC       XR          45428.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 32.0 seconds
